# LaDe-P Demand Forecasting (Hourly, AOI-Level)

## Goal
Develop an AI model to forecast hourly delivery demand per zone (AOI) using the LaDe (Cainiao) last-mile dataset.

## Forecasting Unit
Each observation represents a unique combination of:

- `city`
- `region_id`
- `aoi_id`
- `hour`

### Target Variable
- `demand_count` — number of deliveries within the specified AOI and hour.

In [ ]:
import pandas as pd
import numpy as np

# Visualization (optional)
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Pandas display
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

print("Ready.")

Ready.


In [ ]:
# LaDe time strings do NOT include year; dataset spans 2022-05 to 2022-10
DATA_YEAR = "2022"

# Time bucket granularity
# Pandas is deprecating uppercase 'H'; prefer lowercase 'h'
TIME_BUCKET = "h"   # Hourly

# Split ratios (chronological)
TRAIN_RATIO = 0.60
VAL_RATIO   = 0.20
TEST_RATIO  = 0.20

# Lag/rolling configuration
LAGS = [1, 2, 24, 48, 168]
ROLL_WINDOWS = [24, 168]

# Optional subsetting for prototyping / memory control
# Set DATE_START/DATE_END or MAX_AOIS to None to disable the filter.
DATE_START = "2022-06-01"   # inclusive; set to None to keep full range
DATE_END = "2022-08-31"     # inclusive; set to None to keep full range
MAX_AOIS = 300               # keep the busiest AOIs; set to None for all

In [ ]:
DATA_PATH = "/pickup_cq.csv"
# the path in mac is "downloads/pickup_cq.csv"

df_raw = pd.read_csv(DATA_PATH)
print("Shape:", df_raw.shape)
df_raw.head()

Shape: (1172703, 19)


,order_id,region_id,city,courier_id,accept_time,time_window_start,time_window_end,lng,lat,aoi_id,aoi_type,pickup_time,pickup_gps_time,pickup_gps_lng,pickup_gps_lat,accept_gps_time,accept_gps_lng,accept_gps_lat,ds
0,483671,3,Chongqing,1518,08-14 07:57:00,08-14 09:00:00,08-14 11:00:00,106.46877,29.47204,218,14,08-14 09:38:00,NaN,NaN,NaN,NaN,NaN,NaN,814
1,1746131,3,Chongqing,4706,10-09 07:46:00,10-09 09:00:00,10-09 11:00:00,106.46872,29.47200,218,14,10-09 09:42:00,NaN,NaN,NaN,NaN,NaN,NaN,1009
2,2301722,3,Chongqing,4706,10-09 13:57:00,10-09 13:57:00,10-09 15:57:00,106.46869,29.47191,218,14,10-09 15:53:00,10-09 15:53:00,106.46821,29.46771,10-09 13:56:00,106.46929,29.47231,1009
3,3788723,3,Chongqing,4706,05-19 08:13:00,05-19 11:00:00,05-19 13:00:00,106.46878,29.47208,218,14,05-19 11:59:00,NaN,NaN,NaN,NaN,NaN,NaN,519
4,713435,3,Chongqing,4706,05-22 08:16:00,05-22 09:00:00,05-22 11:00:00,106.46813,29.47228,218,14,05-22 10:40:00,05-22 10:40:00,106.46827,29.47270,NaN,NaN,NaN,522


# Section 4 — Column Selection & Data Dictionary

## Key Columns Used

### 1. Identifiers
- `city` — City name or code  
- `region_id` — Administrative or operational region identifier  
- `aoi_id` — Area of Interest (delivery zone) identifier  
- `aoi_type` — Type/category of AOI  

These define the spatial unit of forecasting.

---

### 2. Time
- `accept_time` — Timestamp when the order was accepted (order arrival time)

This will be transformed into:
- `date`
- `hour`
- `day_of_week`
- etc.

---

### 3. Target Variable (Derived)

- `demand_count` — Number of orders per `(city, region_id, aoi_id, hour)`

Computation logic:

In [4]:
# Missing values in key columns
key_cols = ["city", "region_id", "aoi_id", "aoi_type", "accept_time"]
df_raw[key_cols].isna().sum()

,0
city,0
region_id,0
aoi_id,0
aoi_type,0
accept_time,0


In [5]:
# Basic uniqueness
print("Unique cities:", df_raw["city"].nunique())
print("Unique regions:", df_raw["region_id"].nunique())
print("Unique AOIs:", df_raw["aoi_id"].nunique())
print("Unique couriers:", df_raw["courier_id"].nunique() if "courier_id" in df_raw.columns else "N/A")

Unique cities: 1
Unique regions: 30
Unique AOIs: 4823
Unique couriers: 2982


In [6]:
# Duplicate order IDs (should generally be 0, but check)
dup_orders = df_raw["order_id"].duplicated().sum()
print("Duplicate order_id count:", dup_orders)

Duplicate order_id count: 0


# Section 6 — Time Parsing Assumptions

## Timestamp Format

- `accept_time` is provided in the format:  
  `MM-DD HH:MM:SS`
- No year information is included.

### Assumption
We prepend the year `2022-` to each timestamp before parsing.

Example transformation:
```

"07-15 14:23:10" → "2022-07-15 14:23:10"

```

---

## Hourly Aggregation

We define hourly demand buckets using:

```

floor('H')

```

This assigns each order to the beginning of its corresponding hour.

Example:
```

2022-07-15 14:23:10 → 2022-07-15 14:00:00

```

---

## Demand Definition

- Demand is defined using `accept_time`
- This represents the moment an order enters the system

### Rationale
Using `accept_time` makes the forecast suitable for:
- Dispatch planning
- Courier allocation
- Capacity management

It reflects incoming workload rather than delivery completion.


In [7]:
df = df_raw.copy()

df["accept_dt"] = pd.to_datetime(
    DATA_YEAR + "-" + df["accept_time"].astype(str),
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

print("Unparsed accept_time rows:", df["accept_dt"].isna().sum())
df = df.dropna(subset=["accept_dt"]).copy()

# Optional date-range filtering to reduce data size for prototyping
if DATE_START is not None:
    df = df[df["accept_dt"] >= DATE_START]
if DATE_END is not None:
    end_exclusive = pd.to_datetime(DATE_END) + pd.Timedelta(days=1)
    df = df[df["accept_dt"] < end_exclusive]

print("Time range after filtering:", df["accept_dt"].min(), "→", df["accept_dt"].max())
df[["accept_time", "accept_dt"]].head()

Unparsed accept_time rows: 0
Time range: 2022-04-16 12:33:00 → 2022-10-31 18:46:00


,accept_time,accept_dt
0,08-14 07:57:00,2022-08-14 07:57:00
1,10-09 07:46:00,2022-10-09 07:46:00
2,10-09 13:57:00,2022-10-09 13:57:00
3,05-19 08:13:00,2022-05-19 08:13:00
4,05-22 08:16:00,2022-05-22 08:16:00


# Section 8 — Target Definition via Aggregation Rule

## Hourly Demand Definition

Hourly demand is defined as:

```

demand_count(city, region_id, aoi_id, hour)
= number of orders whose accept_time falls within that hour

```

---

## Aggregation Logic

After parsing timestamps and creating hourly buckets (`bucket_hour`), we compute demand using a group-by aggregation.

### Group Keys
- `city`
- `region_id`
- `aoi_id`
- `aoi_type`
- `bucket_hour`

### Aggregation Operation
- Count number of rows (orders) per group

This produces one row per:

```

(city, region_id, aoi_id, bucket_hour)

```

with:

- `demand_count` as the target variable.

In [8]:
df["bucket_hour"] = df["accept_dt"].dt.floor(TIME_BUCKET)

demand_agg = (
    df.groupby(["city","region_id","aoi_id","aoi_type","bucket_hour"], as_index=False)
      .agg(demand_count=("order_id", "count"))
      .sort_values(["city","region_id","aoi_id","bucket_hour"])
)

print("Aggregated shape:", demand_agg.shape)
demand_agg.head()

/tmp/ipykernel_3116/968307720.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["bucket_hour"] = df["accept_dt"].dt.floor("H")


Aggregated shape: (791132, 6)


,city,region_id,aoi_id,aoi_type,bucket_hour,demand_count
0,Chongqing,3,218,14,2022-05-03 07:00:00,1
1,Chongqing,3,218,14,2022-05-03 14:00:00,1
2,Chongqing,3,218,14,2022-05-06 07:00:00,1
3,Chongqing,3,218,14,2022-05-17 09:00:00,1
4,Chongqing,3,218,14,2022-05-19 07:00:00,2


# Section 10 — Why We Must Add Zero-Demand Hours

## The Problem

If we only keep hours where at least one order exists, the dataset will exclude periods with no activity.

This creates a biased training signal:

- The model only observes **positive demand**
- It never learns what "no demand" looks like
- Forecasts become systematically overestimated

---

## The Solution

We construct a **complete hourly timeline** for every AOI.

For each combination of:

- `city`
- `region_id`
- `aoi_id`
- `aoi_type`

we generate all hourly timestamps within the observed time range.

---

## Zero Filling Rule

If an hour has no recorded orders:


In [9]:
# Optional AOI filtering: keep only the busiest AOIs to control dataset size
if MAX_AOIS is not None:
    aoi_totals = (
        demand_agg.groupby(["city", "region_id", "aoi_id", "aoi_type"])["demand_count"]
        .sum()
        .sort_values(ascending=False)
        .head(MAX_AOIS)
        .reset_index()
    )
    aoi_keys = aoi_totals[["city", "region_id", "aoi_id", "aoi_type"]]
else:
    aoi_keys = demand_agg[["city", "region_id", "aoi_id", "aoi_type"]].drop_duplicates().reset_index(drop=True)

# Global hourly range within (possibly filtered) data
start_hour = demand_agg["bucket_hour"].min()
end_hour = demand_agg["bucket_hour"].max()
full_hours = pd.date_range(start=start_hour, end=end_hour, freq=TIME_BUCKET)

print("AOIs:", len(aoi_keys), "hours per AOI:", len(full_hours))
print("Expected total rows (AOIs * hours):", len(aoi_keys) * len(full_hours))

aoi_keys.head()

Full grid shape: (23100186, 6)


,city,region_id,aoi_id,aoi_type,bucket_hour,demand_count
0,Chongqing,3,218,14,2022-04-16 12:00:00,0
1,Chongqing,3,218,14,2022-04-16 13:00:00,0
2,Chongqing,3,218,14,2022-04-16 14:00:00,0
3,Chongqing,3,218,14,2022-04-16 15:00:00,0
4,Chongqing,3,218,14,2022-04-16 16:00:00,0


In [15]:
def build_aoi_grid(agg, key_row, full_hours):
    mask = (
        (agg["city"] == key_row["city"]) &
        (agg["region_id"] == key_row["region_id"]) &
        (agg["aoi_id"] == key_row["aoi_id"]) &
        (agg["aoi_type"] == key_row["aoi_type"])
    )

    g = agg.loc[mask, ["bucket_hour", "demand_count"]].set_index("bucket_hour")

    # Reindex to full hourly range and fill missing demand with zeros
    g = g.reindex(full_hours)
    g["demand_count"] = g["demand_count"].fillna(0).astype("int16")

    g = g.reset_index().rename(columns={"index": "bucket_hour"})

    g["city"] = key_row["city"]
    g["region_id"] = key_row["region_id"]
    g["aoi_id"] = key_row["aoi_id"]
    g["aoi_type"] = key_row["aoi_type"]

    cols = ["city", "region_id", "aoi_id", "aoi_type", "bucket_hour", "demand_count"]
    return g[cols]

In [16]:
def add_time_and_lag_features(df_aoi):
    df_aoi = df_aoi.sort_values(["city", "region_id", "aoi_id", "bucket_hour"]).copy()

    df_aoi["hour"] = df_aoi["bucket_hour"].dt.hour.astype("int8")
    df_aoi["dow"] = df_aoi["bucket_hour"].dt.dayofweek.astype("int8")
    df_aoi["month"] = df_aoi["bucket_hour"].dt.month.astype("int8")

    df_aoi["is_weekend_fri_sat"] = df_aoi["dow"].isin([4, 5]).astype("int8")
    df_aoi["is_weekend_sat_sun"] = df_aoi["dow"].isin([5, 6]).astype("int8")

    for lag in LAGS:
        df_aoi[f"lag_{lag}"] = df_aoi["demand_count"].shift(lag)

    roll_24, roll_168 = ROLL_WINDOWS
    df_aoi["roll_24_mean"] = df_aoi["demand_count"].shift(1).rolling(roll_24, min_periods=roll_24).mean()
    df_aoi["roll_168_mean"] = df_aoi["demand_count"].shift(1).rolling(roll_168, min_periods=roll_168).mean()

    feature_cols = [f"lag_{lag}" for lag in LAGS] + ["roll_24_mean", "roll_168_mean"]
    df_aoi = df_aoi.dropna(subset=feature_cols)

    return df_aoi

In [ ]:
output_path = "lade_hourly_features.csv"

feature_cols_order = [
    "city", "region_id", "aoi_id", "aoi_type",
    "bucket_hour", "hour", "dow", "month",
    "is_weekend_fri_sat", "is_weekend_sat_sun",
    "demand_count",
] + [f"lag_{lag}" for lag in LAGS] + ["roll_24_mean", "roll_168_mean"]

first_chunk = True

for idx, key_row in aoi_keys.iterrows():
    ts = build_aoi_grid(demand_agg, key_row, full_hours)
    ts = add_time_and_lag_features(ts)

    if ts.empty:
        continue

    ts = ts[feature_cols_order]

    mode = "w" if first_chunk else "a"
    header = first_chunk

    ts.to_csv(output_path, mode=mode, header=header, index=False)

    if first_chunk:
        first_chunk = False

    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(aoi_keys)} AOIs")

print("Saved:", output_path)

In [12]:
# Quick sanity check: read a few rows from the saved CSV (optional)
preview = pd.read_csv(output_path, nrows=5)
preview

demand_full exists: True


In [13]:
print("Finished feature generation and CSV export.")

rows: 23100186
unique AOIs: 4854
hours: 4759
approx rows = AOIs * hours: 23100186


In [ ]:
# Deprecated: model_df is now written incrementally to disk in the streaming loop above.

In [ ]:
# Deprecated: CSV writing was moved into the streaming loop above.